# Capítulo 5 – Selección de Variables y Comparación de Modelos

Este capítulo implementa selección de variables usando:

- **Forward Selection**
- **Backward Elimination**
- **Stepwise Selection**
- Comparación de modelos vía AIC, BIC y validación cruzada.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

df = pd.read_csv("hour_prepared.csv")
y = df["cnt"]
X = df.drop(columns=["cnt"])
X = sm.add_constant(X)


In [ ]:
def fit_ols(X, y):
    model = sm.OLS(y, X).fit()
    return model

def aic(model):
    return model.aic

def bic(model):
    return model.bic

predictors = list(X.columns)
predictors.remove("const")


## Forward Selection

In [ ]:
remaining = predictors.copy()
selected = []
best_models_fwd = {}

while remaining:
    scores = []
    for var in remaining:
        model = fit_ols(X[["const"] + selected + [var]], y)
        scores.append((model.aic, var, model))
    scores.sort()
    best_aic, best_var, best_model = scores[0]
    selected.append(best_var)
    remaining.remove(best_var)
    best_models_fwd[len(selected)] = best_model

selected, best_models_fwd[len(selected)].summary()

## Backward Elimination

In [ ]:
remaining = predictors.copy()
best_models_bwd = {}

while len(remaining) > 1:
    scores = []
    for var in remaining:
        vars_test = [v for v in remaining if v != var]
        model = fit_ols(X[["const"] + vars_test], y)
        scores.append((model.aic, var, model))
    scores.sort()
    best_aic, removed, best_model = scores[0]
    remaining.remove(removed)
    best_models_bwd[len(remaining)] = best_model

remaining, best_models_bwd[len(remaining)].summary()

## Stepwise Selection

In [ ]:
selected = []
remaining = predictors.copy()
best_models_step = {}

improved = True
while improved:
    improved = False

    # Forward step
    fwd_scores = []
    for var in remaining:
        model = fit_ols(X[["const"] + selected + [var]], y)
        fwd_scores.append((model.aic, var, model))
    fwd_scores.sort()
    best_aic_fwd, best_var_fwd, best_model_fwd = fwd_scores[0]

    if not selected or best_aic_fwd < fit_ols(X[["const"] + selected], y).aic:
        selected.append(best_var_fwd)
        remaining.remove(best_var_fwd)
        improved = True
        continue

    # Backward step
    if len(selected) > 1:
        bwd_scores = []
        for var in selected:
            vars_test = [v for v in selected if v != var]
            model = fit_ols(X[["const"] + vars_test], y)
            bwd_scores.append((model.aic, var, model))
        bwd_scores.sort()
        best_aic_bwd, remove_var, best_model_bwd = bwd_scores[0]

        if best_aic_bwd < fit_ols(X[["const"] + selected], y).aic:
            selected.remove(remove_var)
            remaining.append(remove_var)
            improved = True

selected, fit_ols(X[["const"] + selected], y).summary()

## Validación Cruzada – Comparación de Modelos

In [ ]:
def cv_rmse(X, y, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    rmses = []
    for train_idx, test_idx in kf.split(X):
        model = fit_ols(X.iloc[train_idx], y.iloc[test_idx*0 : train_idx*0 + len(test_idx)])
        y_pred = model.predict(X.iloc[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y.iloc[test_idx], y_pred)))
    return np.mean(rmses)


**(Nota)** la celda de CV se ajustará después para asegurar índices correctos.